<a href="https://colab.research.google.com/github/jacquiline18/Jacquiline-CodeBooster-Internship-2026-Phase_01-Data_Engineering-/blob/main/Copy_of_Day_03_ETL_Pandas_APIs_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# PART 2: Weather API ETL Pipeline

## Activity 2 — Call a Live API and Build a Pipeline

### What is an API?
**Level 1:** An API lets one software talk to another. You send a request, it sends back data.

**Level 2 (Waiter Analogy):** You (Python code) tell the waiter (API) what you want. The waiter fetches it from the kitchen (server) and brings it back as JSON.

**Level 3:** REST API — HTTP GET request with parameters → JSON response → parse into Python dict.

---
### Setup: Get a Free API Key
1. Go to: https://openweathermap.org/api
2. Sign up for a free account
3. Go to API keys section → copy your key
4. Paste it in the cell below

**Note:** If you do not have a key yet, a fallback dataset is provided below.

In [ ]:
# ============================================================
# CELL 13 — Configure API settings
# ============================================================

API_KEY = "bd8b79e5d4b6df5d4346c49003f9a571"
# Replace 'YOUR_API_KEY_HERE' with your actual OpenWeatherMap API key
# A free key allows 60 calls per minute — more than enough for this activity

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"
# This is the API endpoint URL
# 'data/2.5/weather' = the specific API route for current weather

# Cities to query — 8 major Indian cities
CITIES = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai',
          'Hyderabad', 'Kolkata', 'Pune', 'Jaipur']
# We will call the API once per city — 8 API calls total

print(f'API configured for {len(CITIES)} cities')
print(f'Cities: {CITIES}')
print('\nIMPORTANT: Replace YOUR_API_KEY_HERE with your actual key before running.')

API configured for 8 cities
Cities: ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Jaipur']

IMPORTANT: Replace YOUR_API_KEY_HERE with your actual key before running.


In [ ]:
# ============================================================
# CELL 14 — EXTRACT: Call API for each city
# ============================================================
import requests;
def fetch_weather(city, API_KEY):
    """
    Fetch current weather data for a given city.
    Returns a dictionary with weather metrics, or None on failure.
    """
    params = {
        'q':     city,      # City name query parameter
        'appid': API_KEY,   # Authentication key
        'units': 'metric'   # Returns temperature in Celsius
    }
    # params is a dictionary — requests will encode it into the URL:
    # ?q=Mumbai&appid=KEY&units=metric

    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        # requests.get() sends an HTTP GET request to BASE_URL
        # timeout=10 — wait max 10 seconds; raise error if no response

        if response.status_code == 200:
            # status_code 200 = HTTP OK = request was successful
            data = response.json()
            # .json() parses the JSON text body into a Python dictionary

            return {
                'city':        city,
                'temperature': round(data['main']['temp'], 1),
                'feels_like':  round(data['main']['feels_like'], 1),
                'humidity':    data['main']['humidity'],
                'pressure':    data['main']['pressure'],
                'wind_speed':  data['wind']['speed'],
                'condition':   data['weather'][0]['description'].title(),
                'visibility':  data.get('visibility', 0) // 1000
                # .get('visibility', 0) — safe access: returns 0 if key missing
                # // 1000 — convert meters to kilometers (integer division)
            }
        else:
            print(f'  ERROR {response.status_code} for {city}: {response.json().get("message","unknown error")}')
            return None

    except requests.exceptions.ConnectionError:
        print(f'  CONNECTION ERROR for {city} — check internet connection')
        return None
    except requests.exceptions.Timeout:
        print(f'  TIMEOUT for {city} — API did not respond in 10 seconds')
        return None
    # try/except handles errors gracefully — one bad city doesn't crash the loop


# Call API for all cities
print('Calling Weather API...')
weather_records = []
# Empty list — we will append one dict per city

for city in CITIES:
    print(f'  Fetching: {city}...', end='')
    record = fetch_weather(city, API_KEY)
    if record:
        weather_records.append(record)
        # .append() adds the dict to our list
        print(f' {record["temperature"]}°C, {record["condition"]}')
    else:
        print(' FAILED')

print(f'\nSuccessfully fetched: {len(weather_records)}/{len(CITIES)} cities')

Calling Weather API...
  Fetching: Mumbai... 31.0°C, Haze
  Fetching: Delhi... 36.0°C, Haze
  Fetching: Bangalore... 24.6°C, Scattered Clouds
  Fetching: Chennai... 31.7°C, Haze
  Fetching: Hyderabad... 30.2°C, Haze
  Fetching: Kolkata... 27.0°C, Haze
  Fetching: Pune... 29.1°C, Scattered Clouds
  Fetching: Jaipur... 37.6°C, Haze

Successfully fetched: 8/8 cities


In [ ]:
# ============================================================
# CELL 15 — Fallback data (if API key not available)
# ============================================================
# Run this cell ONLY if weather_records is empty (API not working)

if len(weather_records) == 0:
    print('Using fallback weather data (API not available)')
    weather_records = [
        {'city':'Mumbai',    'temperature':32.5,'feels_like':36.0,'humidity':78,'pressure':1009,'wind_speed':5.2,'condition':'Partly Cloudy','visibility':8},
        {'city':'Delhi',     'temperature':38.2,'feels_like':41.0,'humidity':35,'pressure':1002,'wind_speed':3.8,'condition':'Clear Sky',    'visibility':10},
        {'city':'Bangalore', 'temperature':26.1,'feels_like':27.0,'humidity':62,'pressure':1016,'wind_speed':2.5,'condition':'Overcast',     'visibility':7},
        {'city':'Chennai',   'temperature':34.8,'feels_like':39.0,'humidity':72,'pressure':1008,'wind_speed':6.1,'condition':'Hazy',         'visibility':5},
        {'city':'Hyderabad', 'temperature':35.4,'feels_like':38.5,'humidity':45,'pressure':1005,'wind_speed':4.2,'condition':'Clear Sky',    'visibility':10},
        {'city':'Kolkata',   'temperature':33.7,'feels_like':37.8,'humidity':80,'pressure':1007,'wind_speed':4.8,'condition':'Humid',        'visibility':6},
        {'city':'Pune',      'temperature':29.3,'feels_like':31.0,'humidity':55,'pressure':1014,'wind_speed':3.1,'condition':'Partly Cloudy','visibility':9},
        {'city':'Jaipur',    'temperature':40.1,'feels_like':43.0,'humidity':22,'pressure':998, 'wind_speed':5.5,'condition':'Sunny',        'visibility':12},
    ]
    print(f'Fallback data loaded for {len(weather_records)} cities')
else:
    print(f'Using live API data for {len(weather_records)} cities')

Using live API data for 8 cities


In [ ]:
# ============================================================
# CELL 16 — TRANSFORM: Build DataFrame from API results
# ============================================================

weather_df = pd.DataFrame(weather_records)
# pd.DataFrame() converts a list of dictionaries into a DataFrame
# Each dictionary becomes one row
# Each dictionary key becomes a column name
# This is the standard pattern for API → DataFrame conversion

print('Weather DataFrame created:')
print(weather_df.to_string(index=False))
print(f'\nShape: {weather_df.shape}')
print(f'Missing values: {weather_df.isnull().sum().sum()}')
print(f'\nData types:')
print(weather_df.dtypes)

Weather DataFrame created:
     city  temperature  feels_like  humidity  pressure  wind_speed        condition  visibility
   Mumbai         31.0        37.6        70      1010        4.12             Haze           3
    Delhi         36.0        34.4        21       999        1.54             Haze           5
Bangalore         24.6        25.2        79      1013        4.47 Scattered Clouds           6
  Chennai         31.7        38.7        73      1008        4.12             Haze           5
Hyderabad         30.2        31.6        51      1008        4.12             Haze           5
  Kolkata         27.0        29.5        78      1006        5.14             Haze           3
     Pune         29.1        29.4        47      1011        4.85 Scattered Clouds          10
   Jaipur         37.6        36.2        20      1003        3.60             Haze           3

Shape: (8, 8)
Missing values: 0

Data types:
city            object
temperature    float64
feels_like     fl

In [ ]:
# ============================================================
# CELL 17 — TRANSFORM: Analysis on weather data
# ============================================================

print('=' * 50)
print('  WEATHER ANALYSIS REPORT — 8 INDIAN CITIES')
print('=' * 50)

# Hottest and coldest city
hottest = weather_df.loc[weather_df['temperature'].idxmax()]
coldest = weather_df.loc[weather_df['temperature'].idxmin()]
# .idxmax() returns the INDEX of the maximum value
# .idxmin() returns the INDEX of the minimum value
# df.loc[index] returns the full row at that index

print(f"\nHottest city : {hottest['city']} at {hottest['temperature']}°C")
print(f"Coldest city : {coldest['city']} at {coldest['temperature']}°C")
print(f"Most humid   : {weather_df.loc[weather_df['humidity'].idxmax()]['city']} "
      f"({weather_df['humidity'].max()}%)")

# Summary statistics
print(f"\nAverage temperature : {weather_df['temperature'].mean():.1f}°C")
print(f"Average humidity    : {weather_df['humidity'].mean():.1f}%")
print(f"Average wind speed  : {weather_df['wind_speed'].mean():.1f} m/s")

# Rankings
print('\nCities ranked by temperature (hottest first):')
ranked = weather_df[['city','temperature','humidity']].sort_values('temperature', ascending=False)
print(ranked.to_string(index=False))

  WEATHER ANALYSIS REPORT — 8 INDIAN CITIES

Hottest city : Jaipur at 37.6°C
Coldest city : Bangalore at 24.6°C
Most humid   : Bangalore (79%)

Average temperature : 30.9°C
Average humidity    : 54.9%
Average wind speed  : 4.0 m/s

Cities ranked by temperature (hottest first):
     city  temperature  humidity
   Jaipur         37.6        20
    Delhi         36.0        21
  Chennai         31.7        73
   Mumbai         31.0        70
Hyderabad         30.2        51
     Pune         29.1        47
  Kolkata         27.0        78
Bangalore         24.6        79


In [ ]:
# ============================================================
# CELL 18 — LOAD: Save weather data to CSV
# ============================================================

weather_df.to_csv('weather_data.csv', index=False)
# Saves the cleaned, structured weather DataFrame to a CSV file
# This file can be loaded into SQLite (Day 2 skills) or used in ML (Day 5)

print('Weather data saved to: weather_data.csv')
print('\nWeather ETL Pipeline: COMPLETE')
print('  EXTRACT   → OpenWeatherMap API called for 8 cities')
print('  TRANSFORM → JSON parsed, DataFrame built, units converted')
print('  LOAD      → weather_data.csv saved')

Weather data saved to: weather_data.csv

Weather ETL Pipeline: COMPLETE
  EXTRACT   → OpenWeatherMap API called for 8 cities
  TRANSFORM → JSON parsed, DataFrame built, units converted
  LOAD      → weather_data.csv saved
